## TEXT ONLY ROBERTA MUSTARD

### install

In [ ]:
!pip -q install transformers datasets evaluate accelerate scikit-learn pandas numpy psutil faster-whisper


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 61.4 MB/s eta 0:00:00


### import

In [ ]:
import os
import gc
import re
import json
import time
import math
import shutil
import psutil
import random
import pathlib
import warnings
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)

from faster_whisper import WhisperModel
from google.colab import files, drive

### settings

In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "distilroberta-base"
OUTPUT_DIR = "/content/mustard_fw_distilroberta"
JSON_PATH = "/content/utterance_asr_stats.json"

TEXT_COL = "whisper_text"
LABEL_COL = "label"
KEY_COL = "key"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-5
WEIGHT_DECAY = 0.01

FW_MODEL_SIZE = "small"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("DEVICE:", DEVICE)
print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)

DEVICE: cuda
MODEL_NAME: distilroberta-base
OUTPUT_DIR: /content/mustard_fw_distilroberta


### download mustard++ but already with faster_whisper transcripts

In [ ]:
uploaded = files.upload()
print(list(uploaded.keys()))

Saving utterance_asr_stats.json to utterance_asr_stats.json
['utterance_asr_stats.json']


### read json

In [ ]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_asr = json.load(f)

print("Items:", len(raw_asr))
raw_asr[0]

Items: 1201


{'key': '1_10004_u',
 'speaker': 'SHELDON',
 'show': 'BBT',
 'sarcasm_label': 0.0,
 'reference_text': 'And of those few months, how long have you been a demented sex pervert?',
 'whisper_text': 'And of those few months, how long have you been a demented sex pervert?',
 'word_error_rate': 0.0,
 'start_of_speech': 0.6399999999999995,
 'end_of_speech': 4.9,
 'detailed_words': [{'word': ' And',
   'start': 0.6399999999999995,
   'end': 1.14,
   'probability': 0.89892578125},
  {'word': ' of', 'start': 1.14, 'end': 1.38, 'probability': 0.9755859375},
  {'word': ' those', 'start': 1.38, 'end': 1.6, 'probability': 0.9990234375},
  {'word': ' few', 'start': 1.6, 'end': 1.9, 'probability': 0.99755859375},
  {'word': ' months,', 'start': 1.9, 'end': 2.3, 'probability': 0.9990234375},
  {'word': ' how', 'start': 2.44, 'end': 2.8, 'probability': 0.99658203125},
  {'word': ' long', 'start': 2.8, 'end': 3.1, 'probability': 1.0},
  {'word': ' have', 'start': 3.1, 'end': 3.34, 'probability': 0.9995117

### prepare dataset

In [ ]:
def prepare_fw_dataframe(raw_items):
    rows = []

    for x in raw_items:
        key = x.get("key")
        whisper_text = x.get("whisper_text")
        reference_text = x.get("reference_text")
        label = x.get("sarcasm_label")
        speaker = x.get("speaker")
        show = x.get("show")
        wer = x.get("word_error_rate")
        start = x.get("start_of_speech")
        end = x.get("end_of_speech")

        if key is None or whisper_text is None or label is None:
            continue

        whisper_text = str(whisper_text).strip()
        if whisper_text == "":
            continue

        try:
            label = int(float(label))
        except:
            continue

        if label not in [0, 1]:
            continue

        rows.append({
            "key": str(key),
            "whisper_text": whisper_text,
            "reference_text": reference_text,
            "label": label,
            "speaker": speaker,
            "show": show,
            "word_error_rate": wer,
            "start_of_speech": start,
            "end_of_speech": end
        })

    df = pd.DataFrame(rows).drop_duplicates(subset=["key"]).reset_index(drop=True)
    return df


df = prepare_fw_dataframe(raw_asr)

print(df.shape)
print(df["label"].value_counts())
df.head()

(1199, 9)
label
1    600
0    599
Name: count, dtype: int64


,key,whisper_text,reference_text,label,speaker,show,word_error_rate,start_of_speech,end_of_speech
0,1_10004_u,"And of those few months, how long have you bee...","And of those few months, how long have you bee...",0,SHELDON,BBT,0.000000,0.64,4.90
1,1_10009_u,Let the dead man talk. So why do you think th...,"Let the dead man talk. So, why do you think that?",0,PENNY,BBT,0.272727,0.00,4.82
2,1_1001_u,What else? Selling on eBay is slightly used.,"What else? Sell it on eBay as ""slightly used.""",0,RAJ,BBT,0.555556,0.00,2.04
3,1_1003_u,to her again good idea sit with her hold her c...,"Good idea, sit with her. Hold her, comfort her...",1,HOWARD,BBT,0.409091,0.00,7.38
4,1_10190_u,"Now that I've given up string theory, I'm stru...","Well, now that I've given up string theory, I'...",0,SHELDON,BBT,0.121212,0.00,10.98


 ### Train / validation / test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (839, 9)
val: (180, 9)
test: (180, 9)


### download split

In [ ]:
train_df.to_csv(os.path.join(OUTPUT_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_split.csv"), index=False)

print("Splits saved")

Splits saved


### Hugging Face Dataset

In [ ]:
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['key', 'whisper_text', 'reference_text', 'label', 'speaker', 'show', 'word_error_rate', 'start_of_speech', 'end_of_speech'],
        num_rows: 839
    })
    validation: Dataset({
        features: ['key', 'whisper_text', 'reference_text', 'label', 'speaker', 'show', 'word_error_rate', 'start_of_speech', 'end_of_speech'],
        num_rows: 180
    })
    test: Dataset({
        features: ['key', 'whisper_text', 'reference_text', 'label', 'speaker', 'show', 'word_error_rate', 'start_of_speech', 'end_of_speech'],
        num_rows: 180
    })
})

### Quality metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

### RAM / GPU / FLOPs / latency functions

In [ ]:
def get_ram_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)


def reset_gpu_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def get_gpu_peak_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None


def measure_classifier_latency(model, tokenizer, texts, batch_size=16, max_length=128):
    model.eval()
    all_preds = []
    all_latencies = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(DEVICE)

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(**inputs)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()

        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy().tolist()
        batch_latency = (end - start) / len(batch_texts)

        all_preds.extend(preds)
        all_latencies.extend([batch_latency] * len(batch_texts))

    return all_preds, all_latencies


def estimate_distilroberta_flops_per_sample(seq_len=128):
    num_layers = 6
    hidden_size = 768
    intermediate_size = 3072

    d = hidden_size
    s = seq_len
    f = intermediate_size

    attention_linear = 4 * s * d * d
    attention_scores = 2 * s * s * d
    ffn = 2 * s * d * f

    flops_per_layer = attention_linear + attention_scores + ffn
    total_flops = num_layers * flops_per_layer

    return int(total_flops)

### Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized = dataset_dict.map(tokenize_batch, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "label"]
tokenized.set_format(type="torch", columns=cols_to_keep)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/839 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['key', 'whisper_text', 'reference_text', 'label', 'speaker', 'show', 'word_error_rate', 'start_of_speech', 'end_of_speech', 'input_ids', 'attention_mask'],
        num_rows: 839
    })
    validation: Dataset({
        features: ['key', 'whisper_text', 'reference_text', 'label', 'speaker', 'show', 'word_error_rate', 'start_of_speech', 'end_of_speech', 'input_ids', 'attention_mask'],
        num_rows: 180
    })
    test: Dataset({
        features: ['key', 'whisper_text', 'reference_text', 'label', 'speaker', 'show', 'word_error_rate', 'start_of_speech', 'end_of_speech', 'input_ids', 'attention_mask'],
        num_rows: 180
    })
})

### model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(DEVICE)

print(type(model))

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'>


### params

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=2
)

### Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

### train and save model

In [ ]:
reset_gpu_peak()
train_ram_before = get_ram_mb()
train_start = time.perf_counter()

trainer.train()

train_end = time.perf_counter()
train_ram_after = get_ram_mb()
train_gpu_peak = get_gpu_peak_mb()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

train_stats = {
    "model_name": MODEL_NAME,
    "train_time_sec": train_end - train_start,
    "train_ram_before_mb": train_ram_before,
    "train_ram_after_mb": train_ram_after,
    "train_gpu_peak_mb": train_gpu_peak
}

print(json.dumps(train_stats, ensure_ascii=False, indent=2))

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.692017,0.690722,0.522222,0.558824,0.211111,0.306452
2,0.677451,0.683727,0.533333,0.520270,0.855556,0.647059
3,0.643779,0.679735,0.561111,0.564706,0.533333,0.548571
4,0.580258,0.685496,0.566667,0.566667,0.566667,0.566667
5,0.546946,0.693355,0.583333,0.588235,0.555556,0.571429


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "model_name": "distilroberta-base",
  "train_time_sec": 134.82653281900002,
  "train_ram_before_mb": 1681.0078125,
  "train_ram_after_mb": 1453.328125,
  "train_gpu_peak_mb": 1347.00390625
}


### preliminary test

In [ ]:
test_texts = test_df[TEXT_COL].tolist()
test_labels = test_df["label"].tolist()

reset_gpu_peak()
infer_ram_before = get_ram_mb()
infer_start = time.perf_counter()

test_preds, latencies = measure_classifier_latency(
    model=model,
    tokenizer=tokenizer,
    texts=test_texts,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

infer_end = time.perf_counter()
infer_ram_after = get_ram_mb()
infer_gpu_peak = get_gpu_peak_mb()

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels,
    test_preds,
    average="binary",
    zero_division=0
)
acc = accuracy_score(test_labels, test_preds)

flops = estimate_distilroberta_flops_per_sample(seq_len=MAX_LENGTH)

method1_stats = {
    "test_accuracy": acc,
    "test_precision": precision,
    "test_recall": recall,
    "test_f1": f1,
    "classifier_infer_total_sec": infer_end - infer_start,
    "classifier_latency_mean_ms": float(np.mean(latencies) * 1000),
    "classifier_latency_std_ms": float(np.std(latencies) * 1000),
    "classifier_ram_before_mb": infer_ram_before,
    "classifier_ram_after_mb": infer_ram_after,
    "classifier_gpu_peak_mb": infer_gpu_peak,
    "approx_flops_per_sample": flops
}

print(json.dumps(method1_stats, ensure_ascii=False, indent=2))

{
  "test_accuracy": 0.6055555555555555,
  "test_precision": 0.564625850340136,
  "test_recall": 0.9222222222222223,
  "test_f1": 0.70042194092827,
  "classifier_infer_total_sec": 0.45238583799982734,
  "classifier_latency_mean_ms": 1.8214389666657858,
  "classifier_latency_std_ms": 0.46027623385255056,
  "classifier_ram_before_mb": 1612.33984375,
  "classifier_ram_after_mb": 1614.35546875,
  "classifier_gpu_peak_mb": 1009.265625,
  "approx_flops_per_sample": 5586812928
}


### download metrics

In [ ]:
all_stats = {}
all_stats.update(train_stats)
all_stats.update(method1_stats)

with open(os.path.join(OUTPUT_DIR, "metrics_method1_asr_transcripts.json"), "w", encoding="utf-8") as f:
    json.dump(all_stats, f, ensure_ascii=False, indent=2)

pd.DataFrame([all_stats]).to_csv(
    os.path.join(OUTPUT_DIR, "metrics_method1_asr_transcripts.csv"),
    index=False
)

test_preds_df = test_df.copy()
test_preds_df["pred"] = test_preds
test_preds_df["correct"] = (test_preds_df["label"] == test_preds_df["pred"]).astype(int)
test_preds_df.to_csv(os.path.join(OUTPUT_DIR, "test_predictions_method1.csv"), index=False)

print("Method 1 results saved")

Method 1 results saved


### get embeddings for hybrid architecture experiment

In [ ]:
def extract_embeddings(texts, tokenizer, base_model, batch_size=16, max_length=128):
    base_model.eval()
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(DEVICE)

        with torch.no_grad():
            outputs = base_model(
                **inputs,
                output_hidden_states=True,
                return_dict=True
            )

        last_hidden = outputs.hidden_states[-1]
        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_hidden = last_hidden * attention_mask
        summed = masked_hidden.sum(dim=1)
        counts = attention_mask.sum(dim=1).clamp(min=1)
        mean_pooled = summed / counts

        all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

### download embeddings

In [ ]:
base_model = model.base_model

train_embeddings = extract_embeddings(
    train_df[TEXT_COL].tolist(),
    tokenizer,
    base_model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

val_embeddings = extract_embeddings(
    val_df[TEXT_COL].tolist(),
    tokenizer,
    base_model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

test_embeddings = extract_embeddings(
    test_df[TEXT_COL].tolist(),
    tokenizer,
    base_model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

np.save(os.path.join(OUTPUT_DIR, "train_embeddings.npy"), train_embeddings)
np.save(os.path.join(OUTPUT_DIR, "val_embeddings.npy"), val_embeddings)
np.save(os.path.join(OUTPUT_DIR, "test_embeddings.npy"), test_embeddings)

print("train:", train_embeddings.shape)
print("val:", val_embeddings.shape)
print("test:", test_embeddings.shape)

train: (839, 768)
val: (180, 768)
test: (180, 768)


### google drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


### save all embeddings and split

In [ ]:
np.save(os.path.join(OUTPUT_DIR, "train_embeddings.npy"), train_embeddings)
np.save(os.path.join(OUTPUT_DIR, "val_embeddings.npy"), val_embeddings)
np.save(os.path.join(OUTPUT_DIR, "test_embeddings.npy"), test_embeddings)

train_df.to_csv(os.path.join(OUTPUT_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_split.csv"), index=False)

print("Embeddings and splits saved to:", OUTPUT_DIR)

Embeddings and splits saved to: /content/mustard_fw_distilroberta


### archive all

In [ ]:
archive_path = shutil.make_archive(
    base_name="/content/mustard_distilroberta_results_faster_whisper",
    format="zip",
    root_dir=OUTPUT_DIR
)

print("Archive created:", archive_path)

Archive created: /content/mustard_distilroberta_results_faster_whisper.zip


### download archive

In [ ]:
files.download("/content/mustard_distilroberta_results_faster_whisper.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## TEST EVALUATION USING WHISPER

### get videos

In [ ]:
!pip -q install PyDrive2

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive_api = GoogleDrive(gauth)

print("Google Drive auth OK")

Google Drive auth OK


In [ ]:
FOLDER_ID = "1xcvGIPGzrcNKkNBwBSoik9sh_SDSH2ov"
DOWNLOAD_DIR = "/content/test_videos_fw"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("FOLDER_ID:", FOLDER_ID)
print("DOWNLOAD_DIR:", DOWNLOAD_DIR)

FOLDER_ID: 1xcvGIPGzrcNKkNBwBSoik9sh_SDSH2ov
DOWNLOAD_DIR: /content/test_videos_fw


In [ ]:
test_keys = test_df[KEY_COL].astype(str).unique().tolist()
needed_filenames = {f"{k}.mp4" for k in test_keys}

print("Unique test keys:", len(test_keys))
print("Need files:", len(needed_filenames))
print(list(sorted(needed_filenames))[:10])

Unique test keys: 180
Need files: 180
['1_10009_u.mp4', '1_10797_u.mp4', '1_10890_u.mp4', '1_11021_u.mp4', '1_11051_u.mp4', '1_11485_u.mp4', '1_11699_u.mp4', '1_11889_u.mp4', '1_12275_u.mp4', '1_1549_u.mp4']


In [ ]:
query = f"'{FOLDER_ID}' in parents and trashed=false"

file_list = drive_api.ListFile({
    "q": query,
    "supportsAllDrives": True,
    "includeItemsFromAllDrives": True
}).GetList()

print("Visible files in folder:", len(file_list))

matched_files = [f for f in file_list if f["title"] in needed_filenames]

print("Matched test videos:", len(matched_files))
print([f["title"] for f in matched_files[:10]])

Visible files in folder: 1203
Matched test videos: 180
['2_495_u.mp4', '2_96_u.mp4', '2_482_u.mp4', '2_506_u.mp4', '1_S12E04_195_u.mp4', '2_234_u.mp4', '2_387_u.mp4', '1_S11E02_299_u.mp4', '1_213_u.mp4', '1_7490_u.mp4']


In [ ]:
downloaded = []
failed = []

for i, f in enumerate(matched_files, 1):
    title = f["title"]
    out_path = os.path.join(DOWNLOAD_DIR, title)

    try:
        if not os.path.exists(out_path):
            f.GetContentFile(out_path)
        downloaded.append(title)

        if i % 20 == 0 or i == len(matched_files):
            print(f"Downloaded {i}/{len(matched_files)}")
    except Exception as e:
        failed.append((title, str(e)))

print("Done")
print("Downloaded:", len(downloaded))
print("Failed:", len(failed))

Downloaded 20/180
Downloaded 40/180
Downloaded 60/180
Downloaded 80/180
Downloaded 100/180
Downloaded 120/180
Downloaded 140/180
Downloaded 160/180
Downloaded 180/180
Done
Downloaded: 180
Failed: 0


In [ ]:
print("Local files:", len(os.listdir(DOWNLOAD_DIR)))
print(os.listdir(DOWNLOAD_DIR)[:10])

Local files: 180
['1_S12E18_303_u.mp4', '1_S11E08_005_u.mp4', '1_S12E07_424_u.mp4', '1_S09E18_357_u.mp4', '2_385_u.mp4', '2_477_u.mp4', '2_248_u.mp4', '1_7395_u.mp4', '2_213_u.mp4', '1_8746_u.mp4']


In [ ]:
VIDEO_EXTS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv"}

def index_video_files(video_root: str):
    video_index = {}
    for root, _, files in os.walk(video_root):
        for fname in files:
            ext = pathlib.Path(fname).suffix.lower()
            if ext in VIDEO_EXTS:
                full_path = os.path.join(root, fname)
                stem = pathlib.Path(fname).stem
                video_index[stem] = full_path
    return video_index

video_index = index_video_files(DOWNLOAD_DIR)

print("Indexed local videos:", len(video_index))
list(video_index.items())[:5]

Indexed local videos: 180


[('1_S12E18_303_u', '/content/test_videos_fw/1_S12E18_303_u.mp4'),
 ('1_S11E08_005_u', '/content/test_videos_fw/1_S11E08_005_u.mp4'),
 ('1_S12E07_424_u', '/content/test_videos_fw/1_S12E07_424_u.mp4'),
 ('1_S09E18_357_u', '/content/test_videos_fw/1_S09E18_357_u.mp4'),
 ('2_385_u', '/content/test_videos_fw/2_385_u.mp4')]

In [ ]:
test_video_df = test_df.copy()
test_video_df["video_id"] = test_video_df[KEY_COL].astype(str)
test_video_df["video_path"] = test_video_df["video_id"].map(video_index)

print("All test rows:", len(test_video_df))
print("Matched local videos:", test_video_df["video_path"].notna().sum())

test_video_df = test_video_df[test_video_df["video_path"].notna()].reset_index(drop=True)
test_video_df.head()

All test rows: 180
Matched local videos: 180


,key,whisper_text,reference_text,label,speaker,show,word_error_rate,start_of_speech,end_of_speech,video_id,video_path
0,2_272_u,We're not peeking?,Weā€™re not peeking?,0,JOEY,FRIENDS,0.333333,0.00,0.98,2_272_u,/content/test_videos_fw/2_272_u.mp4
1,1_7395_u,"If that's all it takes, it's a good thing you ...","Well, if that's all it takes, it's a good thin...",1,AMY,BBT,0.066667,0.42,2.48,1_7395_u,/content/test_videos_fw/1_7395_u.mp4
2,1_S11E10_009_u,"Hey, you had unprotected sex with Howard. You...","(knocking) Hey, you had unprotected sex with H...",0,PENNY,BBT,0.076923,0.98,4.62,1_S11E10_009_u,/content/test_videos_fw/1_S11E10_009_u.mp4
3,1_S12E04_276_u,"So, what do you want to know?","So, what do you want to know?",0,OTHER,BBT,0.000000,1.84,2.68,1_S12E04_276_u,/content/test_videos_fw/1_S12E04_276_u.mp4
4,1_S10E07_247_u,she's enjoyed living with you. It's called bei...,"Uh, she's enjoyed living with you. It's called...",0,PENNY,BBT,0.090909,0.00,1.98,1_S10E07_247_u,/content/test_videos_fw/1_S10E07_247_u.mp4


### get Whisper

In [ ]:
!pip -q install faster-whisper

### faster-whisper model

In [ ]:
from faster_whisper import WhisperModel

compute_type = "float16" if DEVICE == "cuda" else "int8"

fw_model = WhisperModel(
    "small",
    device=DEVICE,
    compute_type=compute_type
)

print("faster-whisper loaded")

faster-whisper loaded


### transcribation func

In [ ]:
def transcribe_video_faster_whisper(video_path, fw_model):
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    segments, info = fw_model.transcribe(
        video_path,
        language="en",
        beam_size=5,
        word_timestamps=False
    )

    text = " ".join(seg.text.strip() for seg in segments).strip()

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()
    elapsed = end - start

    return text, elapsed

### distilroberta classification function

In [ ]:
def predict_text_roberta(text, model, tokenizer, max_length=128):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    ).to(DEVICE)

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model(**inputs)

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    pred = torch.argmax(outputs.logits, dim=-1).item()
    elapsed = end - start

    return pred, elapsed

### test videos: faster-whisper -> DistilRoBERTa

In [ ]:
reset_gpu_peak()
pipeline_ram_before = get_ram_mb()
pipeline_start = time.perf_counter()

y_true = []
y_pred = []

fw_times = []
roberta_times = []
end_to_end_times = []

runtime_transcripts = []

for _, row in test_video_df.iterrows():
    true_label = int(row["label"])
    video_path = row["video_path"]

    t0 = time.perf_counter()

    transcript, fw_time = transcribe_video_faster_whisper(video_path, fw_model)
    pred, roberta_time = predict_text_roberta(
        transcript,
        model,
        tokenizer,
        max_length=MAX_LENGTH
    )

    t1 = time.perf_counter()

    y_true.append(true_label)
    y_pred.append(pred)

    fw_times.append(fw_time)
    roberta_times.append(roberta_time)
    end_to_end_times.append(t1 - t0)

    runtime_transcripts.append(transcript)

pipeline_end = time.perf_counter()
pipeline_ram_after = get_ram_mb()
pipeline_gpu_peak = get_gpu_peak_mb()

print("Processed samples:", len(y_true))

Processed samples: 180


### count metrics

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="binary",
    zero_division=0
)

acc = accuracy_score(y_true, y_pred)

classifier_flops = estimate_distilroberta_flops_per_sample(seq_len=MAX_LENGTH)

pipeline_stats = {
    "n_samples_with_video": len(y_true),

    "pipeline_accuracy": acc,
    "pipeline_precision": precision,
    "pipeline_recall": recall,
    "pipeline_f1": f1,

    "pipeline_total_time_sec": pipeline_end - pipeline_start,

    "faster_whisper_total_time_sec": float(np.sum(fw_times)),
    "faster_whisper_mean_latency_ms": float(np.mean(fw_times) * 1000),
    "faster_whisper_std_latency_ms": float(np.std(fw_times) * 1000),

    "roberta_total_time_sec": float(np.sum(roberta_times)),
    "roberta_mean_latency_ms": float(np.mean(roberta_times) * 1000),
    "roberta_std_latency_ms": float(np.std(roberta_times) * 1000),

    "end_to_end_total_time_sec": float(np.sum(end_to_end_times)),
    "end_to_end_mean_latency_ms": float(np.mean(end_to_end_times) * 1000),
    "end_to_end_std_latency_ms": float(np.std(end_to_end_times) * 1000),

    "pipeline_ram_before_mb": pipeline_ram_before,
    "pipeline_ram_after_mb": pipeline_ram_after,
    "pipeline_gpu_peak_mb": pipeline_gpu_peak,

    "classifier_approx_flops_per_sample": classifier_flops
}

print(json.dumps(pipeline_stats, ensure_ascii=False, indent=2))

{
  "n_samples_with_video": 180,
  "pipeline_accuracy": 0.6055555555555555,
  "pipeline_precision": 0.564625850340136,
  "pipeline_recall": 0.9222222222222223,
  "pipeline_f1": 0.70042194092827,
  "pipeline_total_time_sec": 113.75069306900059,
  "faster_whisper_total_time_sec": 112.07760638999571,
  "faster_whisper_mean_latency_ms": 622.6533688333095,
  "faster_whisper_std_latency_ms": 534.6075089820245,
  "roberta_total_time_sec": 1.4753939100110074,
  "roberta_mean_latency_ms": 8.196632833394485,
  "roberta_std_latency_ms": 15.505274155319395,
  "end_to_end_total_time_sec": 113.70059835401298,
  "end_to_end_mean_latency_ms": 631.6699908556277,
  "end_to_end_std_latency_ms": 537.8795766285974,
  "pipeline_ram_before_mb": 2129.6328125,
  "pipeline_ram_after_mb": 2137.546875,
  "pipeline_gpu_peak_mb": 965.576171875,
  "classifier_approx_flops_per_sample": 5586812928
}


### runtime transcripts

In [ ]:
test_video_results_df = test_video_df.copy()
test_video_results_df["runtime_fw_transcript"] = runtime_transcripts
test_video_results_df["pred"] = y_pred
test_video_results_df["correct"] = (test_video_results_df["label"] == test_video_results_df["pred"]).astype(int)

test_video_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_predictions_method2_runtime_fw_pipeline.csv"),
    index=False
)

with open(os.path.join(OUTPUT_DIR, "metrics_method2_runtime_fw_pipeline.json"), "w", encoding="utf-8") as f:
    json.dump(pipeline_stats, f, ensure_ascii=False, indent=2)

pd.DataFrame([pipeline_stats]).to_csv(
    os.path.join(OUTPUT_DIR, "metrics_method2_runtime_fw_pipeline.csv"),
    index=False
)

print("Method 2 saved")

Method 2 saved


### comparison

In [ ]:
compare_df = test_video_results_df.merge(
    test_df[[KEY_COL, "whisper_text", "reference_text", "label"]],
    on=[KEY_COL, "label"],
    how="left"
)

compare_df.to_csv(
    os.path.join(OUTPUT_DIR, "compare_precomputed_vs_runtime_fw.csv"),
    index=False
)

compare_df[[KEY_COL, "runtime_fw_transcript", "label", "pred"]].head()

,key,runtime_fw_transcript,label,pred
0,2_272_u,We're not peeking?,0,1
1,1_7395_u,"Well, that's all it takes. It's a good thing y...",1,1
2,1_S11E10_009_u,"Hey, you had unprotected sex with Howard, you ...",0,1
3,1_S12E04_276_u,"So, what do you want to know?",0,0
4,1_S10E07_247_u,"She's enjoyed living with you, it's called bei...",0,1


### final

In [ ]:
summary_rows = []

summary_rows.append({
    "pipeline": "distilroberta (precomputed faster-whisper transcripts)",
    "accuracy": method1_stats.get("test_accuracy"),
    "precision": method1_stats.get("test_precision"),
    "recall": method1_stats.get("test_recall"),
    "f1": method1_stats.get("test_f1"),
    "latency_ms": method1_stats.get("classifier_latency_mean_ms"),
    "total_time_sec": method1_stats.get("classifier_infer_total_sec"),
    "memory_mb": max(
        method1_stats.get("classifier_ram_after_mb", 0),
        method1_stats.get("classifier_gpu_peak_mb", 0)
    ),
    "flops_per_sample": method1_stats.get("approx_flops_per_sample")
})

summary_rows.append({
    "pipeline": "faster-whisper (runtime on test videos)",
    "accuracy": None,
    "precision": None,
    "recall": None,
    "f1": None,
    "latency_ms": pipeline_stats.get("faster_whisper_mean_latency_ms"),
    "total_time_sec": pipeline_stats.get("faster_whisper_total_time_sec"),
    "memory_mb": pipeline_stats.get("pipeline_gpu_peak_mb"),
    "flops_per_sample": None
})

summary_rows.append({
    "pipeline": "distilroberta (runtime faster-whisper transcripts)",
    "accuracy": pipeline_stats.get("pipeline_accuracy"),
    "precision": pipeline_stats.get("pipeline_precision"),
    "recall": pipeline_stats.get("pipeline_recall"),
    "f1": pipeline_stats.get("pipeline_f1"),
    "latency_ms": pipeline_stats.get("roberta_mean_latency_ms"),
    "total_time_sec": pipeline_stats.get("roberta_total_time_sec"),
    "memory_mb": pipeline_stats.get("pipeline_gpu_peak_mb"),
    "flops_per_sample": pipeline_stats.get("classifier_approx_flops_per_sample")
})

summary_rows.append({
    "pipeline": "faster-whisper + distilroberta (e2e)",
    "accuracy": pipeline_stats.get("pipeline_accuracy"),
    "precision": pipeline_stats.get("pipeline_precision"),
    "recall": pipeline_stats.get("pipeline_recall"),
    "f1": pipeline_stats.get("pipeline_f1"),
    "latency_ms": pipeline_stats.get("end_to_end_mean_latency_ms"),
    "total_time_sec": pipeline_stats.get("pipeline_total_time_sec"),
    "memory_mb": pipeline_stats.get("pipeline_gpu_peak_mb"),
    "flops_per_sample": None
})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(os.path.join(OUTPUT_DIR, "final_experiment_summary.csv"), index=False)
print("Final summary saved")

,pipeline,accuracy,precision,recall,f1,latency_ms,total_time_sec,memory_mb,flops_per_sample
0,distilroberta (precomputed faster-whisper tran...,0.605556,0.564626,0.922222,0.700422,1.821439,0.452386,1614.355469,5.586813e+09
1,faster-whisper (runtime on test videos),NaN,NaN,NaN,NaN,622.653369,112.077606,965.576172,NaN
2,distilroberta (runtime faster-whisper transcri...,0.605556,0.564626,0.922222,0.700422,8.196633,1.475394,965.576172,5.586813e+09
3,faster-whisper + distilroberta (e2e),0.605556,0.564626,0.922222,0.700422,631.669991,113.750693,965.576172,NaN


Final summary saved


### dowmload

In [ ]:
archive_path = shutil.make_archive(
    base_name="/content/mustard_fw_distilroberta_results",
    format="zip",
    root_dir=OUTPUT_DIR
)

print("Archive created:", archive_path)

Archive created: /content/mustard_fw_distilroberta_results.zip


In [ ]:
files.download("/content/mustard_fw_distilroberta_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>